

---

# Surface material detection using Full Scene imagery

In [1]:
PROMPT = """Act as a civil engineering material analyst specializing in road surfaces. Your task is to analyze the provided Google Street View image and accurately identify the dominant road surface material present
Your analysis must follow these steps to ensure accuracy:
1.  **Identify Road Surface Area**: Clearly delineate the primary road surface in the image, ignoring sidewalks, shoulders, or surrounding terrain.
2.  **Visual Evidence Extraction**: For the identified road surface, describe the visual cues that indicate its material type. Focus specifically on:
    *   **Texture & Micro-structure**: Look for characteristics such as:
        *   **Paved**: Smooth, uniform, often dark or light grey. May show aggregate, cracks, patches, or lane markings. If asphalt, a granular, somewhat coarse texture. If concrete, a finer, often broom-finished texture with expansion joints.
        *   **Gravel**: Loose, irregularly shaped stones of varying sizes with visible gaps and an uneven surface. Often exhibits tire tracks or displacement.
        *   **Mud**: Soft, wet, or dried soil, often with ruts, puddles, or deep impressions from vehicles. Can vary widely in color and consistency.
        *   **Dirt**: Unpaved, dry, compacted soil. Less uniform than paved, but more stable than mud, often exhibiting dust or tire marks.
    *   **Reflectance & Specularity**: Observe how light interacts with the surface. Is it matte (dirt, some mud), somewhat reflective (wet mud, new asphalt), or does it show clear highlights (wet paved roads, standing water)?
    *   **Contextual Cues**: Consider environmental factors such as surrounding vegetation, drainage, and road infrastructure (e.g., presence of road signs, guardrails nearby, but not directly on the road surface).
**Output Format**:
Provide your findings in a structured JSON format:
{
  "road_surface_material": "[Material classification: Paved (Asphalt/Concrete), Gravel, Mud, Dirt, or Other]",
  "confidence_score": "[0-100%]",
  "visual_reasoning": "[1-3 sentences describing specific visual evidence supporting the classification, e.g., 'Surface exhibits uniform dark gray color with visible aggregate and clear lane markings, consistent with asphalt pavement.']",
  "image_url": "{image_url}"
}
**Note:**
**Important Considerations:**
*   Focus exclusively on the material directly comprising the main driving surface.
*   Ignore temporary conditions like standing water or debris unless they are definitive indicators of the underlying road material.
*   If the material is ambiguous or mixed, classify based on the dominant type and note ambiguity in reasoning."""

In [2]:
import pandas_gbq
import vertexai
from vertexai.preview.generative_models import GenerativeModel, Part
import urllib.parse

# Query the BigQuery table
project_id = "YOUR_PROJECT_ID" #@param {type:"string"} #@param {type:"string"}
track_id = 't1:YrRl38Z-yzI95yarpLV3Fw:5001ee' #@param {type:"string"}
dataset_id = 'imagery_insights___us' #@param {type:"string"} #@param {type:"string"}
urls_table_name = 'urls_new' #@param {type:"string"}

sql_query = f"""
SELECT
  t2.signedUrl
FROM
  `{project_id}`.`{dataset_id}`.tracks_unnested AS t1
INNER JOIN
  `{project_id}`.`{dataset_id}`.`{urls_table_name}` AS t2
ON
  t1.observation2 = t2.observationId
WHERE
  t1.trackId = '{track_id}'
"""
df = pandas_gbq.read_gbq(sql_query, project_id, dialect="standard")

# Get the GCS URL
if not df.empty:
    gcs_url = df['signedUrl'].iloc[0]
    print(f"GCS URL (from BigQuery): {gcs_url}")

    # Decode the GCS URL fully first
    fully_decoded_gcs_url = urllib.parse.unquote(gcs_url)
    print(f"Fully Decoded GCS URL: {fully_decoded_gcs_url}")

    # Convert to gs:// URI for Vertex AI directly from fully decoded GCS URL
    if fully_decoded_gcs_url.startswith("https://storage.googleapis.com/"):
        gcs_uri_for_vertex_ai = fully_decoded_gcs_url.replace("https://storage.googleapis.com/", "gs://")
    elif fully_decoded_gcs_url.startswith("https://storage.mtls.cloud.google.com/"):
        gcs_uri_for_vertex_ai = fully_decoded_gcs_url.replace("https://storage.mtls.cloud.google.com/", "gs://")
    else:
        gcs_uri_for_vertex_ai = fully_decoded_gcs_url

    print(f"Final GCS URI for Vertex AI: {gcs_uri_for_vertex_ai}")

    try:
        # Initialize Vertex AI
        vertexai.init(project=project_id, location="global")
        model_name = "gemini-3.5-flash" #@param {type:"string"}
        model = GenerativeModel(model_name)

        # Prepare the prompt and image for the model using Part.from_uri()
        prompt = PROMPT 
        image_part = Part.from_uri(uri=gcs_uri_for_vertex_ai, mime_type="image/jpeg")

        # Send to Gemini model
        response = model.generate_content([image_part, prompt])
        print("Gemini Model Response:")
        print(response.text)

    except Exception as e:
        print(f"An unexpected error occurred: {e}")
else:
    print("No URL found for the given trackId.")

/opt/homebrew/lib/python3.13/site-packages/google/cloud/aiplatform/models.py:52: FutureWarning: Support for google-cloud-storage < 3.0.0 will be removed in a future version of google-cloud-aiplatform. Please upgrade to google-cloud-storage >= 3.0.0.
  from google.cloud.aiplatform.utils import gcs_utils


/opt/homebrew/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2689: UserWarning: A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.
  record_batch = self.to_arrow(
/opt/homebrew/lib/python3.13/site-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


GCS URL (from BigQuery): https://storage.mtls.cloud.google.com/everything_imagery/full_scene_home_depot/o1:YrRl38Z-yzI95yarpLV3Fw_2:5001ee.jpg
Fully Decoded GCS URL: https://storage.mtls.cloud.google.com/everything_imagery/full_scene_home_depot/o1:YrRl38Z-yzI95yarpLV3Fw_2:5001ee.jpg
Final GCS URI for Vertex AI: gs://everything_imagery/full_scene_home_depot/o1:YrRl38Z-yzI95yarpLV3Fw_2:5001ee.jpg


Gemini Model Response:
```json
{
  "road_surface_material": "Paved (Asphalt)",
  "confidence_score": "95%",
  "visual_reasoning": "The road surface displays a weathered, dark-to-medium grey color with extensive networks of fine cracks and joint lines, characteristic of aged asphalt pavement. Its texture is relatively uniform and firm, showing no signs of loose gravel, mud, or uncompacted dirt, and is integrated with concrete curbs on the residential street corner.",
  "image_url": "https://images.cocodataset.org/val2017/000000397133.jpg"
}
```
